In [3]:
# ==============================================================================
# ATIVIDADE 2: CHATBOT VERSÃO 2 - DECISION TREE
# ==============================================================================

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# ==============================================================================
# 1. CARREGAR DATASET
# ==============================================================================

df = pd.read_csv(
    'dataset_moveis_100.csv'
)


# ==============================================================================
# 2. DIVISÃO DOS DADOS EM TREINO E TESTE
# ==============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.30,
    random_state=42,
    stratify=df['intencao']
)


# ==============================================================================
# 3. CRIAR A PIPELINE COM TF-IDF + DECISION TREE
# ==============================================================================

pipeline_tree = Pipeline([

    (
        'vectorizer',
        TfidfVectorizer(
            ngram_range=(1, 2)
        )
    ),

    (
        'classifier',
        DecisionTreeClassifier(
            random_state=42,
            max_depth=10,
            min_samples_leaf=2
        )
    )
])


# ==============================================================================
# 4. TREINAR A PIPELINE
# ==============================================================================

pipeline_tree.fit(
    X_train,
    y_train
)


# ==============================================================================
# 5. FAZER AS PREVISÕES
# ==============================================================================

y_pred = pipeline_tree.predict(
    X_test
)


# ==============================================================================
# 6. EXIBIR A MATRIZ DE CONFUSÃO
# ==============================================================================

print(
    "\n=== MATRIZ DE CONFUSÃO - DECISION TREE ==="
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ==============================================================================
# 7. EXIBIR O RELATÓRIO DE CLASSIFICAÇÃO
# ==============================================================================

print(
    "\n=== RELATÓRIO DE CLASSIFICAÇÃO - DECISION TREE ==="
)

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ==============================================================================
# 8. TESTES MANUAIS
# ==============================================================================

LIMIAR_CONFIANCA = 0.70


print(
    "\n=== INICIANDO BATERIA DE TESTES "
    "(8 INPUTS OBRIGATÓRIOS) ==="
)


for i in range(1, 9):

    print(
        f"\n[Teste {i}/8]"
    )


    # --------------------------------------------------------------------------
    # Solicitar frase do usuário
    # --------------------------------------------------------------------------

    frase = input(
        "Digite a frase do cliente: "
    ).strip()


    # --------------------------------------------------------------------------
    # Verificar se a entrada está vazia
    # --------------------------------------------------------------------------

    if not frase:

        print(
            "Entrada inválida. "
            "Digite uma solicitação para continuar."
        )

        continue


    # --------------------------------------------------------------------------
    # Verificar se a frase possui poucas palavras
    # --------------------------------------------------------------------------

    quantidade_palavras = len(
        frase.split()
    )


    if quantidade_palavras < 2:

        print(
            "Não foi possível identificar a solicitação. "
            "Digite uma frase mais detalhada."
        )

        continue


    # --------------------------------------------------------------------------
    # Obter probabilidades
    # --------------------------------------------------------------------------

    probs = pipeline_tree.predict_proba(
        [frase]
    )


    maior_prob = np.max(
        probs
    )


    # --------------------------------------------------------------------------
    # Obter intenção prevista
    # --------------------------------------------------------------------------

    intencao = pipeline_tree.predict(
        [frase]
    )[0]


    # --------------------------------------------------------------------------
    # Aplicar regra de confiança
    # --------------------------------------------------------------------------

    if maior_prob >= LIMIAR_CONFIANCA:

        print(
            f"Intenção identificada: {intencao}"
        )

        print(
            f"Probabilidade: {maior_prob * 100:.2f}%"
        )

    else:

        print(
            "Não foi possível identificar a intenção "
            "com segurança."
        )

        print(
            "Encaminhando você para um atendente humano..."
        )



=== MATRIZ DE CONFUSÃO - DECISION TREE ===
[[5 0 1 0 0]
 [1 3 2 0 0]
 [0 0 6 0 0]
 [0 2 2 2 0]
 [0 0 3 0 3]]

=== RELATÓRIO DE CLASSIFICAÇÃO - DECISION TREE ===
                    precision    recall  f1-score   support

logistica_entregas       0.83      0.83      0.83         6
       reclamacoes       0.60      0.50      0.55         6
           suporte       0.43      1.00      0.60         6
 trocas_devolucoes       1.00      0.33      0.50         6
            vendas       1.00      0.50      0.67         6

          accuracy                           0.63        30
         macro avg       0.77      0.63      0.63        30
      weighted avg       0.77      0.63      0.63        30


=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===

[Teste 1/8]
Digite a frase do cliente: Comprar
Não foi possível identificar a solicitação. Digite uma frase mais detalhada.

[Teste 2/8]
Digite a frase do cliente: Qual o saldo da minha conta?
Não foi possível identificar a intenção 